In [10]:
import json
import pandas as pd
from datasets import load_dataset

In [11]:
TARGET_COUNT = 3000
MIN_WORDS = 15
MAX_WORDS = 200

In [12]:
def is_clean_prompt(sample):
    """
    Filters for single-turn, safe, and complex user prompts.
    """
    
    if sample.get('toxic') is True:
        return False
    
    if sample.get('language') != 'English':
        return False
        
    conversation = sample.get('conversation')
    if not isinstance(conversation, list) or len(conversation) == 0:
        return False
        
    first_turn = conversation[0]
    if first_turn.get('role') != 'user':
        return False
        
    content = first_turn.get('content', '')
    
    word_count = len(content.split())
    if word_count < MIN_WORDS or word_count > MAX_WORDS:
        return False
        
    return True

In [13]:
dataset = load_dataset("allenai/WildChat-1M", split="train", streaming=True)

clean_wildchat_data = []
seen_prompts = set()
for i, sample in enumerate(dataset):
    if is_clean_prompt(sample):
        user_prompt = sample['conversation'][0]['content']
        if user_prompt in seen_prompts:
            continue
        seen_prompts.add(user_prompt)
        clean_wildchat_data.append({
            "wildchat_id": sample.get('conversation_hash'),
            "clean_prompt": user_prompt
        })
        if len(clean_wildchat_data) % 100 == 0:
            print(f"Collected {len(clean_wildchat_data)}...")
    if len(clean_wildchat_data) >= TARGET_COUNT:
        break

Collected 100...
Collected 200...
Collected 300...
Collected 400...
Collected 500...
Collected 600...
Collected 700...
Collected 800...
Collected 900...
Collected 1000...
Collected 1100...
Collected 1200...
Collected 1300...
Collected 1400...
Collected 1500...
Collected 1600...
Collected 1700...
Collected 1800...
Collected 1900...
Collected 2000...
Collected 2100...
Collected 2200...
Collected 2300...
Collected 2400...
Collected 2500...
Collected 2600...
Collected 2700...
Collected 2800...
Collected 2900...
Collected 3000...


In [14]:
clean_wildchat_data

[{'wildchat_id': 'cf1267ca6b2f6fccc9c36652a00059a1',
  'clean_prompt': 'Old age PT hx of DM, HTN, dyslipidemia His ECG I.II, aVF (MI) what is the highest risk \n\nfactor for this condition?'},
 {'wildchat_id': '4b8016c129dfae2f09d646059c8496b7',
  'clean_prompt': 'Help me flesh out the following boss fights for a third person action game. Give them movesets, they use against the player in the encounter:\n\nAoko "The Ronin". He is a giant of a man, working as a butcher. He is aggressive and often ruthless. As a powerhouse, he lack technique and instead focuses on raw strength and force.\n\nGao "The Ninja". He is a handsome loner. Little is known about him, other than his impeccable fashion sense. He is always dressed in a form fitting suit. He is a balanced character with an answer for most situations.\n\nNajlina "The Kunoichi" She is a very attractive young woman. She likes showing off her shapely body in her revealing outfit. She is sensual and alluring. She can apply intense grapplin

In [15]:
df_wildchat = pd.DataFrame(clean_wildchat_data)
df_wildchat.head()

,wildchat_id,clean_prompt
0,cf1267ca6b2f6fccc9c36652a00059a1,"Old age PT hx of DM, HTN, dyslipidemia His ECG..."
1,4b8016c129dfae2f09d646059c8496b7,Help me flesh out the following boss fights fo...
2,0a342f2a33d8486f338dbc3881323da3,Create boss fights for an action-packed video ...
3,e8481c51c544ae7a94c248505d0b3413,Give the following characters a moveset for a ...
4,aa7c3f49343e097be66442288abd1dac,"Let A, B, and C be events with\n\nProb[A] = 0...."


In [16]:
len(df_wildchat)

3000

In [17]:
with open('wildchat_filtered.jsonl', 'w', encoding='utf-8') as f:
    for entry in clean_wildchat_data:
        f.write(json.dumps(entry) + '\n')

print(f"Saved {len(clean_wildchat_data)} prompts to wildchat_filtered.jsonl")

Saved 3000 prompts to wildchat_filtered.jsonl


In [18]:
df_wildchat[df_wildchat.wildchat_id == '383ddd881190f7939eaa934357fa1e6a']['clean_prompt'].values[0]

'Переведи дословно весь текст.A total of six rockets were launched from Syria towards Israel, and three crossed into Israeli territory, the Israel Defense Forces (IDF) said. One of the rockets landed in the Israeli-occupied Golan Heights.\n\nSo far the IDF has not reported any damage in Israeli territory.\n\nIt is the latest flare-up after Israel struck Palestinian militant targets in southern Lebanon and Gaza early Friday in response to dozens of rockets fired from Lebanon into Israeli territory.'